# NextTick — Phase 1: Data Pipeline

## Objective

Build a clean, reproducible data pipeline that:
1. Fetches 5 years of daily stock data for 10 diversified tickers
2. Engineers 6 technical indicators as predictive features
3. Creates targets for two modeling tasks:
   - **Classification:** Will the stock close UP or DOWN tomorrow?
   - **Regression:** By what percentage will the price change tomorrow?
4. Exports a single modeling-ready CSV for downstream notebooks

## Tickers

Diversified across sectors to capture varied market behavior:
- **Tech:** AAPL, MSFT, GOOGL, AMZN, TSLA
- **Finance:** JPM, GS, BAC
- **Healthcare:** JNJ
- **Energy:** XOM

## Output

`data/processed/nexttick_dataset.csv` — feature-engineered, labeled dataset ready for modeling in notebooks `02` through `05`.

## 1. Environment Setup

Import core libraries and verify versions. 

In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import pandas_ta as ta
import sklearn
import torch

print("pandas:    ", pd.__version__)
print("numpy:     ", np.__version__)
print("yfinance:  ", yf.__version__)
print("pandas_ta: ", ta.version)
print("sklearn:   ", sklearn.__version__)
print("torch:     ", torch.__version__)

pandas:     3.0.2
numpy:      2.2.6
yfinance:   1.3.0
pandas_ta:  0.4.71b0
sklearn:    1.8.0
torch:      2.11.0+cpu


## 2. Inspecting the Data Source

Before building a pipeline, fetch a single ticker and inspect what `yfinance` actually returns. 

### Understanding the columns

| Column | Meaning |
|---|---|
| **Open** | Price when market opened (9:30 AM ET) |
| **High** | Highest price during the day |
| **Low** | Lowest price during the day |
| **Close** | Price when market closed (4:00 PM ET) |
| **Volume** | Number of shares traded |

### Why `auto_adjust=True`

We use the flag `auto_adjust=True` so that `Close` is automatically adjusted for stock splits and dividends. Without this adjustment, corporate actions would appear in the data as massive fake "price drops," poisoning any model trained on the raw series. Adjusted Close reflects the actual economic experience of a shareholder.

In [2]:
# Re-fetch AAPL with auto-adjust enabled for simplicity
# Now "Close" = adjusted close (handles splits/dividends automatically)
aapl = yf.download(
    "AAPL",
    period="5y",
    interval="1d",
    auto_adjust=True  # key flag: Close is now split/dividend-adjusted
)

# Handle the MultiIndex columns that yfinance now returns
if isinstance(aapl.columns, pd.MultiIndex):
    aapl.columns = aapl.columns.get_level_values(0)

print("Shape:", aapl.shape)
print("\nDate range:", aapl.index.min().date(), "→", aapl.index.max().date())
print("\nColumns:", list(aapl.columns))
print("\nFirst 3 rows:")
print(aapl.head(3))
print("\nLast 3 rows:")
print(aapl.tail(3))
print("\nMissing values per column:")
print(aapl.isna().sum())

[*********************100%***********************]  1 of 1 completed

Shape: (1255, 5)

Date range: 2021-04-21 → 2026-04-20

Columns: ['Close', 'High', 'Low', 'Open', 'Volume']

First 3 rows:
Price            Close        High         Low        Open    Volume
Date                                                                
2021-04-21  130.028427  130.271926  127.885640  128.918073  68847100
2021-04-22  128.509003  130.661525  127.992786  129.580389  84566500
2021-04-23  130.827072  131.606257  128.723238  128.723238  78657500

Last 3 rows:
Price            Close        High         Low        Open    Volume
Date                                                                
2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100
2026-04-17  270.230011  272.299988  266.720001  266.959991  61436200
2026-04-20  273.049988  274.274994  270.290009  270.329987  34667241

Missing values per column:
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


## 3. Building the Reusable Pipeline

Rather than writing ad-hoc cells for each step, we wrap the full pipeline in four composable functions. 

### Pipeline stages

`fetch_ticker_data` -> `add_features` -> `add_targets` -> `dropna` -> tag with ticker

### Features engineered

| Feature | Purpose |
|---|---|
| `daily_return` | % change from previous close - the raw signal |
| `sma_10`, `sma_20` | Short/long moving averages -> trend direction |
| `volatility_10` | Rolling std of returns -> market regime |
| `momentum_10` | 10-day price change -> trend strength |
| `rsi_14` | Relative Strength Index (0–100) -> overbought/oversold |

### Targets created

Using `.shift(-1)` to place *tomorrow's* outcome on *today's* feature row. 

- `target_return` - tomorrow's daily return (regression target)
- `target_direction` - 1 if tomorrow closes UP, 0 if DOWN (classification target)

> Features and targets must be computed *per ticker* before combining. Computing rolling features on a vertically stacked multi-ticker DataFrame would blend prices across different stocks and silently corrupt the data.

In [3]:
def fetch_ticker_data(ticker: str, period: str = "5y") -> pd.DataFrame:
    """
    Download OHLCV data for a single ticker.
    Returns a DataFrame indexed by Date with columns: Close, High, Low, Open, Volume.
    Close is already split/dividend-adjusted (auto_adjust=True).
    """
    df = yf.download(ticker, period=period, interval="1d", auto_adjust=True, progress=False)
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def add_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add technical indicators as new columns:
    daily_return, sma_10, sma_20, volatility_10, momentum_10, rsi_14.
    """
    df = df.copy()
    df["daily_return"] = df["Close"].pct_change()
    df["sma_10"] = df["Close"].rolling(window=10).mean()
    df["sma_20"] = df["Close"].rolling(window=20).mean()
    df["volatility_10"] = df["daily_return"].rolling(window=10).std()
    df["momentum_10"] = df["Close"].pct_change(periods=10)
    df["rsi_14"] = ta.rsi(df["Close"], length=14)
    return df


def add_targets(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add prediction targets as new columns:
      target_return     — tomorrow's daily return (regression target)
      target_direction  — 1 if tomorrow is UP, 0 if DOWN (classification target)
    Uses shift(-1) to look exactly one day into the future.
    """
    df = df.copy()
    df["target_return"] = df["daily_return"].shift(-1)
    df["target_direction"] = (df["target_return"] > 0).astype(int)
    return df


def process_ticker(ticker: str, period: str = "5y") -> pd.DataFrame:
    """
    Full single-ticker pipeline: fetch → features → targets → drop NaNs → tag.
    Returns a clean, modeling-ready DataFrame with a 'ticker' column added.
    """
    df = fetch_ticker_data(ticker, period=period)
    df = add_features(df)
    df = add_targets(df)
    df = df.dropna().copy()
    df["ticker"] = ticker
    return df


# Sanity check: re-run the AAPL pipeline through the new functions and confirm
# the output matches what we already validated cell-by-cell above.
aapl_refactored = process_ticker("AAPL", period="5y")
print(f"AAPL refactored pipeline: {len(aapl_refactored)} rows, {len(aapl_refactored.columns)} columns")
print(f"Columns: {list(aapl_refactored.columns)}")
print(f"Date range: {aapl_refactored.index.min().date()} → {aapl_refactored.index.max().date()}")
print(f"\nLast 3 rows:")
print(aapl_refactored.tail(3))

AAPL refactored pipeline: 1235 rows, 14 columns
Columns: ['Close', 'High', 'Low', 'Open', 'Volume', 'daily_return', 'sma_10', 'sma_20', 'volatility_10', 'momentum_10', 'rsi_14', 'target_return', 'target_direction', 'ticker']
Date range: 2021-05-18 → 2026-04-17

Last 3 rows:
Price            Close        High         Low        Open    Volume  \
Date                                                                   
2026-04-15  266.429993  266.559998  257.809998  258.160004  49913500   
2026-04-16  263.399994  267.160004  261.269989  266.799988  43323100   
2026-04-17  270.230011  272.299988  266.720001  266.959991  61436200   

Price       daily_return      sma_10      sma_20  volatility_10  momentum_10  \
Date                                                                           
2026-04-15      0.029363  258.823997  254.649500       0.013938     0.049805   
2026-04-16     -0.011373  259.600996  255.322499       0.014815     0.030395   
2026-04-17      0.025930  261.031998  256.38

## 4. Scaling to the All Tickers

Apply the pipeline to all 10 tickers independently, then concatenate. Each ticker gets its own `ticker` column so the model can later distinguish rows belonging to different stocks.

A `try/except` block around each ticker prevents one bad download from crashing the entire run.

In [4]:
# Define the ticker universe — diversified across sectors per your spec
TICKERS = [
    "AAPL", "MSFT", "GOOGL", "AMZN", "TSLA",   # Tech
    "JPM", "GS", "BAC",                         # Finance
    "JNJ", "XOM",                               # Healthcare + Energy
]

# Process each ticker independently
all_ticker_dfs = []
for ticker in TICKERS:
    try:
        df_t = process_ticker(ticker, period="5y")
        print(f"{ticker:6s} — {len(df_t):5d} rows, "
              f"UP rate: {df_t['target_direction'].mean():.3f}")
        all_ticker_dfs.append(df_t)
    except Exception as e:
        print(f"{ticker:6s} — FAILED: {e}")

# Combine into one big DataFrame
combined_df = pd.concat(all_ticker_dfs, ignore_index=False)

print(f"\n")
print(f"Combined dataset: {len(combined_df)} rows across {combined_df['ticker'].nunique()} tickers")
print(f"Overall UP rate: {combined_df['target_direction'].mean():.3f}")
print(f"Per-ticker row counts:")
print(combined_df["ticker"].value_counts())

AAPL   —  1235 rows, UP rate: 0.531
MSFT   —  1235 rows, UP rate: 0.519
GOOGL  —  1235 rows, UP rate: 0.534
AMZN   —  1235 rows, UP rate: 0.514
TSLA   —  1235 rows, UP rate: 0.518
JPM    —  1235 rows, UP rate: 0.534
GS     —  1235 rows, UP rate: 0.527
BAC    —  1235 rows, UP rate: 0.509
JNJ    —  1235 rows, UP rate: 0.514
XOM    —  1235 rows, UP rate: 0.535


Combined dataset: 12350 rows across 10 tickers
Overall UP rate: 0.524
Per-ticker row counts:
ticker
AAPL     1235
MSFT     1235
GOOGL    1235
AMZN     1235
TSLA     1235
JPM      1235
GS       1235
BAC      1235
JNJ      1235
XOM      1235
Name: count, dtype: int64


## 5. Saving the Processed Dataset

Write the combined DataFrame to `data/processed/nexttick_dataset.csv`. All downstream modeling notebooks (02–05) read from this file.

In [5]:
import os
from pathlib import Path

# Define output path — relative to the project root
output_dir = Path("../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "nexttick_dataset.csv"

# Save — keep the Date index so chronological order is preserved
combined_df.to_csv(output_file, index=True)

# Verify the save worked
file_size_mb = os.path.getsize(output_file) / (1024 * 1024)
print(f"   Saved: {output_file}")
print(f"   File size: {file_size_mb:.2f} MB")
print(f"   Rows: {len(combined_df):,}")
print(f"   Columns: {len(combined_df.columns)}")

# Sanity check: reload and verify
reloaded = pd.read_csv(output_file, index_col="Date", parse_dates=True)
print(f"\n   Reload test: {len(reloaded)} rows, matches original: {len(reloaded) == len(combined_df)}")
print(f"\nFirst 2 rows of reloaded data:")
print(reloaded.head(2))

   Saved: ..\data\processed\nexttick_dataset.csv
   File size: 2.82 MB
   Rows: 12,350
   Columns: 14

   Reload test: 12350 rows, matches original: True

First 2 rows of reloaded data:
                 Close        High         Low        Open    Volume  \
Date                                                                   
2021-05-18  121.809906  123.897796  121.741611  123.478267  63342900   
2021-05-19  121.653801  121.878196  119.868360  120.161058  92612000   

            daily_return      sma_10      sma_20  volatility_10  momentum_10  \
Date                                                                           
2021-05-18     -0.011246  123.583905  126.454544       0.016407    -0.021806   
2021-05-19     -0.001282  123.272400  126.035814       0.016350    -0.024967   

               rsi_14  target_return  target_direction ticker  
Date                                                           
2021-05-18  28.148546      -0.001282                 0   AAPL  
2021-05-19  

## 6. Summary

### Dataset produced
- **12,350 rows** across **10 tickers** × **~1,235 trading days** each
- **14 columns:** 5 OHLCV + 6 features + 2 targets + 1 ticker tag
- **Zero missing values** after cleanup
- **Date range:** May 2021 → April 2026

### Baseline to beat
The overall **UP rate is 52.4%** - meaning a naive model that predicts "UP every day" would already achieve ~52.4% accuracy. Any classification model we build must beat this to be considered useful. This is a known macro pattern in US equities, not a labeling artifact.

### Class balance note
The mild UP-majority (~52%) is consistent across all 10 tickers (range: 50.9% – 53.5%). We will address this imbalance in downstream models using `class_weight='balanced'` where available, and by reporting F1-score alongside accuracy so the metric isn't dominated by the majority class.

### Next: Phase 2 — Baseline Models
In `02_baseline_models.ipynb` we load `nexttick_dataset.csv` and establish the performance floor using:
- **Logistic Regression** (classification)
- **Linear Regression** (regression)
- **`TimeSeriesSplit`** cross-validation - not standard k-fold - to respect chronological order
- **`StandardScaler`** - fit on train only, never on the full dataset, to avoid leakage